# Mulimodal CAD Edit Benchmark Data Wrangler

Pull edits and requests from database and convert into pandas dataframes for analysis.

In [ ]:
import os
import sys
module_path = os.path.abspath(os.path.join('../..'))
sys.path.append(module_path)
from src.utils.db import DatabaseManager
from src.utils.process_config import load_config
from src.utils.visualise_results import parse_rating


import pandas as pd

### Load data from database

In [ ]:
config_path = os.path.join(module_path, 'src', 'config', 'edit_v2.json')

class Args:
    def __init__(self, config):
        self.config = config
args = Args(config=config_path)
config = load_config(args.config)
dbm = DatabaseManager(config)
print("Database manager initialized from config:", config_path)
dbm.print_db_schema_counts()
dbm.print_db_summary(count_limits={"ratings": 50})

# check how many requests have two edits
request_2_count = dict()
for edit in dbm.edits.find({}):
    req_id = edit["request"]
    if req_id not in request_2_count:
        request_2_count[req_id] = 0
    request_2_count[req_id] += 1

request_counts = {}
for req_id, count in request_2_count.items():
    if count not in request_counts:
        request_counts[count] = 0
    request_counts[count] += 1
print("Request edit counts:", request_counts)

# check how many edits do not point to a valid request
invalid_request_edits = 0
for edit in dbm.edits.find({}):
    req_id = edit["request"]
    request = dbm.requests.find_one({"_id": req_id})
    if request is None:
        print(f"Edit {edit['_id']} has invalid request reference {req_id}")
        invalid_request_edits += 1
print("Number of edits with invalid request reference:", invalid_request_edits)


#### Create requests dataframe

In [ ]:
# process videos to get duration of edits and requests
import subprocess

request_videos = {}
requests = dbm.requests.find({})
for request in requests:
    video_key = "30_720_audio"
    if video_key in request:
        video_path = os.path.join(dbm.root_dir, request[video_key])

        if os.path.exists(video_path):
            result = subprocess.run(
                ["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "default=noprint_wrappers=1:nokey=1", video_path],
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
            )
            duration = float(result.stdout)
            request_videos[request["_id"]] = duration

for request_id, duration in request_videos.items():
    print(f"Request ID: {request_id}, Video Duration: {duration:.2f} seconds")

In [ ]:
# Create a dataframe of requests, adding 'is_human' from the user db

requests_data = []
request_videos = {}


for request in dbm.requests.find({}):
    request_dict = dict(request)
    
    # Lookup is_human from the users collection, if 'user' field exists
    user_id = request.get("user")
    is_human = None
    if user_id is not None:
        user_doc = dbm.users.find_one({"_id": user_id})
        if user_doc is not None:
            is_human = user_doc.get("is_human")
    request_dict["is_human"] = is_human

    video_key = "30_720_audio"
    if video_key in request:
        video_path = os.path.join(dbm.root_dir, request[video_key])

        if os.path.exists(video_path):
            result = subprocess.run(
                ["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "default=noprint_wrappers=1:nokey=1", video_path],
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
            )
            duration = float(result.stdout)
            request_videos[request["_id"]] = duration

    # add request_video_duration column
    request_dict["request_video_duration"] = request_videos.get(request["_id"], None)

    
    requests_data.append(request_dict)
 
df_requests = pd.DataFrame(requests_data)
print(f"Created dataframe with {len(df_requests)} requests")
print(f"Columns: {list(df_requests.columns)}")
df_requests.head()

In [ ]:
# run statistics on CAD models input and output

import cadquery as cq

def cad_stats(step_file):
    shape = cq.importers.importStep(step_file)

    if hasattr(shape, 'val'):
        occ_shape = shape.val()
    else:
        occ_shape = shape

    return {
        "num_faces": len(occ_shape.Faces()),
        "num_edges": len(occ_shape.Edges()),
        "num_vertices": len(occ_shape.Vertices())
    }

def stats_from_brep_id(dbm, brep_id):
    brep = dbm.breps.find_one({"_id": brep_id})
    step_file = brep.get("step", [None])[0]
    if not step_file:
        print(f"No STEP file found for BREP ID {brep_id}")
        return None
    step_path = os.path.join(dbm.root_dir, step_file)
    if not os.path.exists(step_path):
        print(f"STEP file not found at path: {step_path}")
        return None
    stats = cad_stats(step_path)
    return stats

request_accumulator = {}
for request in dbm.requests.find({}):
    brep = dbm.breps.find_one({"_id": request["brep_start"]})
    request_accumulator[request["_id"]] = stats_from_brep_id(dbm, brep["_id"])
num_requests = float(len(request_accumulator))
total_faces = sum(stats["num_faces"] for stats in request_accumulator.values()) / num_requests
total_edges = sum(stats["num_edges"] for stats in request_accumulator.values()) / num_requests
total_vertices = sum(stats["num_vertices"] for stats in request_accumulator.values()) / num_requests
print(f"Request Average faces: {total_faces}, Average edges: {total_edges}, Average vertices: {total_vertices}")

gt_edit_accumulator = {}
other_human_edit_accumulator = {}
for edit in dbm.edits.find({}):
    request = dbm.requests.find_one({"_id": edit["request"]})
    brep = dbm.breps.find_one({"_id": edit["brep_end"]})
    
    request_user = dbm.users.find_one({"_id": request["user"]})
    edit_user = dbm.users.find_one({"_id": edit["user"]})

    if not edit_user["is_human"]:
        continue

    stats = stats_from_brep_id(dbm, brep["_id"])
    if edit["user"] == request["user"]:
        gt_edit_accumulator[request["_id"]] = stats
    else:
        other_human_edit_accumulator[request["_id"]] = stats

num_gt_edits = float(len(gt_edit_accumulator))
num_other_edits = float(len(other_human_edit_accumulator))
gt_avg_faces = sum(stats["num_faces"] for stats in gt_edit_accumulator.values()) / num_gt_edits
gt_avg_edges = sum(stats["num_edges"] for stats in gt_edit_accumulator.values()) / num_gt_edits
gt_avg_vertices = sum(stats["num_vertices"] for stats in gt_edit_accumulator.values()) / num_gt_edits
other_avg_faces = sum(stats["num_faces"] for stats in other_human_edit_accumulator.values()) / num_other_edits
other_avg_edges = sum(stats["num_edges"] for stats in other_human_edit_accumulator.values()) / num_other_edits
other_avg_vertices = sum(stats["num_vertices"] for stats in other_human_edit_accumulator.values()) / num_other_edits
print(f"GT Edit Average faces: {gt_avg_faces}, Average edges: {gt_avg_edges}, Average vertices: {gt_avg_vertices}")
print(f"Other Human Edit Average faces: {other_avg_faces}, Average edges: {other_avg_edges}, Average vertices: {other_avg_vertices}")

#### Create dataframe of edits

In [ ]:
# Create a dataframe of edits with their corresponding request information, ratings, and is_human column
edits_data = []
for edit in dbm.edits.find({}):
    edit_dict = dict(edit)
    request_id = edit.get("request")

    # Get the corresponding request
    request = dbm.requests.find_one({"_id": request_id})

    if request:
        # Add request fields with 'request_' prefix to avoid collision
        for key, value in request.items():
            edit_dict[f"request_{key}"] = value

    # Lookup is_human from the users collection for the edit's user (not the request's user)
    edit_user_id = edit.get("user")
    is_human = None
    if edit_user_id is not None:
        user_doc = dbm.users.find_one({"_id": edit_user_id})
        if user_doc is not None:
            is_human = user_doc.get("is_human")
    edit_dict["is_human"] = is_human

    # Add ratings/scores for this edit
    private_scores_by_suffix = {}
    for rating in dbm.ratings.find({"edit": edit["_id"], "user": {"$ne": "private.us-west-2.0fe3ef0902668b13"}}):
        parsed = parse_rating(rating)
        if not parsed:
            continue
        for metric_name, score in parsed.items():
            if metric_name.startswith("private."):
                suffix = metric_name.rsplit("_", 1)[-1] if "_" in metric_name else "rating"
                private_scores_by_suffix.setdefault(suffix, []).append(score)
            else:
                edit_dict[metric_name] = score
    for suffix, values in private_scores_by_suffix.items():
        valid = [v for v in values if v is not None and v == v]
        # Take median as long as there's at least one valid value
        if len(valid) >= 1:
            edit_dict[f"rating_avg_human_{suffix}"] = float(pd.Series(valid).median())
            edit_dict[f"rating_mean_human_{suffix}"] = float(pd.Series(valid).mean())
            edit_dict[f"rating_median_human_{suffix}"] = float(pd.Series(valid).median())
        else:
            edit_dict[f"rating_avg_human_{suffix}"] = 1.0/6.0 if suffix == "instruction" else 0.0
            edit_dict[f"rating_mean_human_{suffix}"] = 1.0/6.0 if suffix == "instruction" else 0.0
            edit_dict[f"rating_median_human_{suffix}"] = 1.0/6.0 if suffix == "instruction" else 0.0

    # Compute cost: $60/hr for humans, token_counts.cost_estimate for models
    if is_human:
        st = edit.get("start_time")
        et = edit.get("end_time")
        # add edit_duration column
        # edit_dict["edit_duration"] = et - st
        if st is not None and et is not None:
            duration_secs = float(et) - float(st)
            edit_dict["cost"] = 60.0 * (duration_secs / 3600.0)
        else:
            edit_dict["cost"] = None
    else:
        tc = edit.get("token_counts")
        if isinstance(tc, dict) and "cost_estimate" in tc:
            edit_dict["cost"] = tc["cost_estimate"]
        else:
            edit_dict["cost"] = None

    # Add request (input) CAD stats for all edits
    req_stats = request_accumulator.get(request_id) or {}
    edit_dict["request_num_faces"] = req_stats.get("num_faces")
    edit_dict["request_num_edges"] = req_stats.get("num_edges")
    edit_dict["request_num_vertices"] = req_stats.get("num_vertices")

    # Add output CAD stats for human edits only
    output_stats = None
    if is_human and request:
        if edit.get("user") == request.get("user"):
            output_stats = gt_edit_accumulator.get(request_id)
        else:
            output_stats = other_human_edit_accumulator.get(request_id)
    output_stats = output_stats or {}
    edit_dict["output_num_faces"] = output_stats.get("num_faces")
    edit_dict["output_num_edges"] = output_stats.get("num_edges")
    edit_dict["output_num_vertices"] = output_stats.get("num_vertices")

    # gt_human: True if this human made the request, False if other human, NaN otherwise
    if is_human and request:
        edit_dict["gt_human"] = edit.get("user") == request.get("user")
    else:
        edit_dict["gt_human"] = None

    # Difference between output and input geometry (NaN if either side is missing)
    r = edit_dict.get("request_num_faces")
    o = edit_dict.get("output_num_faces")
    edit_dict["delta_num_faces"] = (o - r) if (r is not None and o is not None) else None
    r = edit_dict.get("request_num_edges")
    o = edit_dict.get("output_num_edges")
    edit_dict["delta_num_edges"] = (o - r) if (r is not None and o is not None) else None
    r = edit_dict.get("request_num_vertices")
    o = edit_dict.get("output_num_vertices")
    edit_dict["delta_num_vertices"] = (o - r) if (r is not None and o is not None) else None

    edits_data.append(edit_dict)

df_edits = pd.DataFrame(edits_data)
print(f"Created dataframe with {len(df_edits)} edits")
print(f"Columns: {list(df_edits.columns)}")

# Show which rating columns were added
rating_cols = [c for c in df_edits.columns if any(kw in c.lower() for kw in ["similarity", "instruction", "quality", "iou", "clip", "dino", "chamfer", "threshold"])]
print(f"\nRating columns ({len(rating_cols)}): {rating_cols}")
print("Non-null counts per rating column:")
# for col in rating_cols:
#     print(f"  {col}: {df_edits[col].notna().sum()} / {len(df_edits)}")

df_edits.head()

#### Create dataframe of ratings

Ratings can come from human raters or VLMs.

In [ ]:
# set default values for missing ratings (these differ for quality and instruction following)
df_edits.rating_avg_human_quality = df_edits.rating_avg_human_quality.fillna(0.0)
df_edits.rating_avg_human_instruction = df_edits.rating_avg_human_instruction.fillna(1.0/6.0)

In [ ]:
from src.utils.visualise_results import parse_rating

In [ ]:
# Create a tidy df_ratings: one row per (edit, rater, metric) for inter-rater reliability & metric correlation
ratings_rows = []
for edit in dbm.edits.find({}):
    edit_id = edit["_id"]
    edit_user_id = edit.get("user")
    request_id = edit.get("request")

    request = dbm.requests.find_one({"_id": request_id})
    if not request:
        continue

    # Edit-level context
    user_doc = dbm.users.find_one({"_id": edit_user_id})
    edit_is_human = user_doc.get("is_human") if user_doc else None
    gt_human = (edit_user_id == request.get("user")) if edit_is_human else None

    # Request-level context
    ctx = {
        "edit_id": edit_id,
        "edit_user": edit_user_id,
        "is_human": edit_is_human,
        "gt_human": gt_human,
        "request_id": request_id,
        "request_type": request.get("request_type"),
        "difficulty": request.get("difficulty"),
        "modality": request.get("modality"),
        "assembly": request.get("assembly"),
        "parametric": request.get("parametric"),
        "source": request.get("source"),
        "filename_cleaned": request.get("filename_cleaned"),
        "brep_start": request.get("brep_start"),
    }

    for rating in dbm.ratings.find({"edit": edit_id, "user": {"$ne": "private.us-west-2.0fe3ef0902668b13"}}):
        rater_id = rating.get("user")

        # Determine rater type
        if rater_id == "similarity_eval":
            rater_type = "geometric"
        elif "private." in str(rater_id):
            rater_type = "human"
        elif all(k in str(rater_id) for k in ["gpt", "rating"]):
            rater_type = "vlm"
        else:
            rater_type = "other"

        parsed = parse_rating(rating)
        if not parsed:
            continue

        for metric_name, score in parsed.items():
            # For private raters, strip the user prefix to get a clean metric name
            if rater_type == "human" and "_" in metric_name:
                metric = metric_name.rsplit("_", 1)[-1]
            elif rater_type == "vlm" and "_" in metric_name:
                metric = metric_name.rsplit("_", 1)[-1]
            else:
                metric = metric_name

            ratings_rows.append({
                **ctx,
                "rater_id": rater_id,
                "rater_type": rater_type,
                "metric": metric,
                "score": score,
            })

df_ratings = pd.DataFrame(ratings_rows)
print(f"Created df_ratings with {len(df_ratings)} rows")
print(f"Rater types: {df_ratings.rater_type.value_counts().to_dict()}")
print(f"Metrics: {df_ratings.metric.value_counts().to_dict()}")
print(f"Unique raters: {df_ratings.rater_id.nunique()}")
df_ratings.head(10)

In [ ]:
instruction_threshold = (5.0-1.0) / 6.0
quality_threshold = (5.0-1.0) / 6.0

df_edits["human_acceptance_post_avg"] = ((df_edits["rating_avg_human_instruction"] >= instruction_threshold) & (df_edits["rating_avg_human_quality"] >= quality_threshold)).astype(int)
# df_edits["vlm_acceptance_post_avg"] = ((df_edits["rating_avg_vlm_instruction"] >= instruction_threshold) & (df_edits["rating_avg_vlm_quality"] >= quality_threshold)).astype(int)

In [ ]:
df_human_ratings = df_ratings[df_ratings.rater_type == 'human'].copy()
df_human_ratings['instruction_threshold'] = instruction_threshold
df_human_ratings['quality_threshold'] = quality_threshold
df_human_ratings['instruction_threshold_met'] = df_human_ratings['score'] >= instruction_threshold
df_human_ratings['quality_threshold_met'] = df_human_ratings['score'] >= quality_threshold
df_human_ratings['human_acceptance'] = df_human_ratings['instruction_threshold_met'] & df_human_ratings['quality_threshold_met']
df_human_acceptance = df_human_ratings.groupby('edit_id')['human_acceptance'].apply(lambda x: x.fillna(0).mean()).reset_index()

# same for vlm
df_vlm_ratings = df_ratings[df_ratings.rater_type == 'vlm'].copy()
df_vlm_ratings['instruction_threshold'] = instruction_threshold
df_vlm_ratings['quality_threshold'] = quality_threshold
df_vlm_ratings['instruction_threshold_met'] = df_vlm_ratings['score'] >= instruction_threshold
df_vlm_ratings['quality_threshold_met'] = df_vlm_ratings['score'] >= quality_threshold
df_vlm_ratings['vlm_acceptance'] = df_vlm_ratings['instruction_threshold_met'] & df_vlm_ratings['quality_threshold_met']
df_vlm_acceptance = df_vlm_ratings.groupby('edit_id')['vlm_acceptance'].apply(lambda x: x.fillna(0).mean()).reset_index()


In [ ]:
# merge df_human_acceptance and df_vlm_acceptance into df_edits
# Note: 'edit id is just id for df_edit', so join on 'id' not 'edit_id'.
df_edits = pd.merge(df_edits, df_human_acceptance, left_on='_id', right_on='edit_id', how='left')
df_edits = pd.merge(df_edits, df_vlm_acceptance, left_on='_id', right_on='edit_id', how='left')
# fill NaN with 0 for acceptance columns
df_edits['human_acceptance'] = df_edits['human_acceptance'].fillna(0)
df_edits['vlm_acceptance'] = df_edits['vlm_acceptance'].fillna(0)
df_edits

In [ ]:
# export dataframes to csv

date = "2026_03_18"

df_requests.to_csv("../../consolidated_data/edits_v2_df_requests_{}.csv".format(date), index=False)
df_edits.to_csv("../../consolidated_data/edits_v2_df_edits_{}.csv".format(date), index=False)
df_ratings.to_csv("../../consolidated_data/edits_v2_df_ratings_{}.csv".format(date), index=False)